# ITO5201 – Assessment 1: Section 2
## Probability
**Student:** Johannes Coetzee  
**Student Number:** 36384852

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

DEBUG = False  # Set to True to enable debug output

`numpy` is imported as `np` for all numerical array operations and random-number generation. `matplotlib.pyplot` provides the plotting interface. `DEBUG = False` is a module-level gate: every `print(...) if DEBUG else None` statement in the notebook is silenced when `False`, enabling clean output for submission while preserving the ability to re-enable step-by-step logging by flipping a single variable.

---
## Question 4 – Bayes Rule
### Q4.I – Simulate the Box/Fruit Experiment
*GitHub issue: #17*

Box contents:
- **Red**: 3 apples, 5 oranges
- **Blue**: 4 apples, 4 oranges
- **Yellow**: 1 apple, 1 orange

In [ ]:


class Box:
    def __init__(self, name):
        self.name = name
        self.total_fruits = 0
        self.fruit_types = []
        self.contents = {}
        self.fruit_weights = np.array([])

    def sample_fruit(self, rng):
        return rng.choice(self.fruit_types, p=self.fruit_weights)
    
    def add_fruit(self, fruit, count=1):
        if fruit in self.contents:
            self.contents[fruit] += count
        else:
            self.contents[fruit] = count
        self.total_fruits = sum(self.contents.values())
        self.fruit_types = list(self.contents.keys())
        self.fruit_weights = np.array(list(self.contents.values())) / self.total_fruits

    def remove_fruit(self, fruit, count=1):
        if fruit in self.contents and self.contents[fruit] >= count:
            self.contents[fruit] -= count
            if self.contents[fruit] == 0:
                del self.contents[fruit]
            self.total_fruits = sum(self.contents.values())
            self.fruit_types = list(self.contents.keys())
            self.fruit_weights = np.array(list(self.contents.values())) / self.total_fruits
        else:
            raise ValueError(f"Not enough {fruit} to remove.")
        
    def get_fruit_count(self, fruit):
        return self.contents.get(fruit, 0)


Red box contents: {'apple': 3, 'orange': 5}
Blue box contents: {'apple': 4, 'orange': 4}
Yellow box contents: {'apple': 1, 'orange': 1}
Chosen box: red
Chosen fruit: orange
Chosen box: yellow
Chosen fruit: orange
Chosen box: red
Chosen fruit: apple
Chosen box: yellow
Chosen fruit: orange
['red' 'yellow' 'red' 'yellow']
['orange' 'orange' 'apple' 'orange']


In [ ]:
# SETUP: Create boxes and add fruits

red_box = Box("red")
blue_box = Box("blue")
yellow_box = Box("yellow")
    
red_box.add_fruit("apple", 3)
red_box.add_fruit("orange", 5)
blue_box.add_fruit("apple", 4)
blue_box.add_fruit("orange", 4)
yellow_box.add_fruit("apple", 1)
yellow_box.add_fruit("orange", 1)

print(f"Red box contents: {red_box.contents}") if DEBUG else None
print(f"Blue box contents: {blue_box.contents}") if DEBUG else None
print(f"Yellow box contents: {yellow_box.contents}") if DEBUG else None



In [ ]:
# FUNCTION: Simulate selecting a box and a fruit

def fruit_experiment(n, rng=None) -> tuple[np.ndarray, np.ndarray]:
    """
    Simulate selecting a box uniformly at random, then a fruit from that box.
    Returns arrays of selected boxes and fruits for n repetitions.

    Parameters:
    - n: number of repetitions
    - rng: optional random number generator (default is None, which uses np.random.default_rng

    Returns:
    - boxes: array of selected box names
    - fruits: array of selected fruit names
    """

    if rng is None:
        rng = np.random.default_rng()
    # YOUR CODE HERE
    boxes = []
    fruits = []
    box_list = [red_box, blue_box, yellow_box]
    for _ in range(n):
        chosen_box = rng.choice(box_list)
        print(f"Chosen box: {chosen_box.name}") if DEBUG else None
        boxes.append(chosen_box.name)
        fruits.append(chosen_box.sample_fruit(rng))
        print(f"Chosen fruit: {fruits[-1]}") if DEBUG else None
    return np.array(boxes), np.array(fruits)



In [ ]:
# EXECUTION: Run the fruit experiment with 4 repetitions and a fixed random seed
rng = np.random.default_rng(42)
boxes, fruits = fruit_experiment(4, rng)
print(boxes)
print(fruits)

**`__init__`** initialises an empty box with the given name. `total_fruits` tracks the running total count. `fruit_types` is a list of fruit names in insertion order. `contents` is a plain dict mapping each fruit name to its count. `fruit_weights` is a numpy array of sampling probabilities that `rng.choice` consumes directly via its `p=` argument.

**`add_fruit`** increments the fruit's count if it already exists, or inserts it with the given count otherwise. It then recomputes `total_fruits`, `fruit_types`, and `fruit_weights` from `contents` from scratch — dividing each count by `total_fruits` ensures the weights always sum exactly to 1.

**`remove_fruit`** checks that sufficient quantity exists before decrementing. If the count reaches zero, the key is deleted from `contents` so that `rng.choice` never encounters a zero-probability entry. It then rebuilds `total_fruits`, `fruit_types`, and `fruit_weights` identically to `add_fruit`. Raises `ValueError` if removal is not possible.

**`get_fruit_count`** uses `dict.get(key, 0)` to safely return 0 for a fruit type that was never added, avoiding a `KeyError`.

**`sample_fruit`** delegates to `rng.choice(self.fruit_types, p=self.fruit_weights)`, drawing a single fruit name proportionally to the current inventory counts.

The three global instances (`red_box`, `blue_box`, `yellow_box`) are created immediately and stocked to match the problem specification (3 apples + 5 oranges, 4 + 4, and 1 + 1 respectively).

**`fruit_experiment`** selects `n` box–fruit pairs. `rng.choice(box_list)` picks a box uniformly at random (no `p=` argument means equal probability), then `chosen_box.sample_fruit(rng)` draws a fruit according to that box's inventory weights. Both results are accumulated and returned as numpy string arrays.

### Q4.II – Derive P(yellow | apple) Using Bayes' Rule
*GitHub issue: #18*

**Formal derivation:**

Let $B \in \{\text{red}, \text{blue}, \text{yellow}\}$ denote the selected box and $F \in \{\text{apple}, \text{orange}\}$ the selected fruit.

**Prior** (uniform box selection):
$$P(B = b) = \frac{1}{3} \quad \text{for all } b$$

**Likelihoods** $P(F = \text{apple} \mid B = b)$:

$$P(F = \text{apple} \mid B = \text{red}) = \frac{3}{8}$$

$$P(F = \text{apple} \mid B = \text{blue}) = \frac{4}{8} = \frac{1}{2}$$

$$P(F = \text{apple} \mid B = \text{yellow}) = \frac{1}{2}$$

**Marginal** $P(F = \text{apple})$ via total probability theorem:

$$P(F = \text{apple}) = \sum_{b} P(F = \text{apple} \mid B = b) \cdot P(B = b)$$

$$= \frac{3}{8} \cdot \frac{1}{3} + \frac{1}{2} \cdot \frac{1}{3} + \frac{1}{2} \cdot \frac{1}{3}$$

$$= \frac{3}{24} + \frac{4}{24} + \frac{4}{24} = \frac{11}{24}$$

**Posterior** via Bayes' Rule:
$$P(B = \text{yellow} \mid F = \text{apple}) = \frac{P(F = \text{apple} \mid B = \text{yellow}) \cdot P(B = \text{yellow})}{P(F = \text{apple})}$$

$$= \frac{\dfrac{1}{2} \cdot \dfrac{1}{3}}{\dfrac{11}{24}} = \frac{\dfrac{1}{6}}{\dfrac{11}{24}} = \frac{1}{6} \cdot \frac{24}{11} = \frac{4}{11} \approx 0.364$$

### Verification Against Simulation
*GitHub issue: #19*

In [ ]:
# Verify P(yellow | apple) derivation against simulation — #19
rng = np.random.default_rng(0)
boxes, fruits = fruit_experiment(100_000, rng)

# Empirical P(yellow | apple)
apple_indices      = np.where(fruits == "apple")[0]
yellow_given_apple = np.sum(boxes[apple_indices] == "yellow") / len(apple_indices)

print(f"Empirical  P(yellow | apple) : {yellow_given_apple:.4f}")
print(f"Analytical P(yellow | apple) : {4/11:.4f}  (= 4/11)")
print(f"Difference                   : {abs(yellow_given_apple - 4/11):.4f}  "
      f"({'within' if abs(yellow_given_apple - 4/11) < 0.01 else 'outside'} expected statistical margin)")

`fruit_experiment(100_000, rng)` runs 100,000 simulated trials. The empirical posterior $\hat{P}(\text{yellow} \mid \text{apple})$ is computed from the raw arrays:
* `np.where(fruits == "apple")[0]` — returns the integer indices of all trials where an apple was drawn.
* `boxes[apple_indices] == "yellow"` — boolean mask over that subset, `True` wherever the yellow box was selected.
* `np.sum(...) / len(apple_indices)` — the fraction of apple-draws that came from the yellow box, i.e., the empirical conditional probability.

The absolute difference against the analytical value $4/11 \approx 0.3636$ is expected to be well under 0.01 for $N = 100{,}000$ by the law of large numbers.

---
## Question 5 – Expected Values
### Q5.I – Implement `die_experiment`
*GitHub issue: #20*

**Game rules:**
- $X$ = outcome of first die roll (1–6)
- $Y(i)$ = outcome of $i$-th subsequent roll if $i \leq X$, else 0
- $Z = \sum_{i=1}^{6} Y(i)$ = player score

In [15]:
def die_experiment(n_repetitions, rng=None):
    """
    Simulate the one-player die game for n_repetitions rounds.
    Returns array of scores Z for each repetition.
    """
    if rng is None:
        rng = np.random.default_rng()
    
    scores = []
    for _ in range(n_repetitions):
        X = rng.integers(1, 7)           # first roll
        Z = rng.integers(1, 7, size=X).sum()  # sum of X subsequent rolls
        scores.append(Z)
    return np.array(scores)

rng = np.random.default_rng(42) # Initialize the random number generator with a fixed seed for reproducibility
# Example usage of the die_experiment function
print("Die experiment results for 10 repetitions:")
scores = die_experiment(10, rng)
print(f"Scores: {scores}")

Die experiment results for 10 repetitions:
Scores: [ 5 13 18 21  9 24  7 19 18 23]


**`die_experiment`** simulates `n_repetitions` rounds of the game:
* `rng.integers(1, 7)` — draws one integer uniformly from $\{1,\ldots,6\}$. The upper bound is exclusive in numpy's `integers`, which is why 7 is passed instead of 6.
* `rng.integers(1, 7, size=X)` — draws all $X$ subsequent rolls in a single call, returning a 1-D array of length $X$. Using `size=X` is preferred over a Python loop `for _ in range(X): roll = rng.integers(1,7)` because numpy executes the inner loop in C, which is orders of magnitude faster for large $n$.
* `.sum()` — totals the $X$ rolls to produce the score $Z$.

An `rng is None` guard defaults to a fresh `np.random.default_rng()` if no generator is supplied, ensuring the function is still usable without explicit seeding. Callers that need reproducibility pass their own seeded generator.

### Q5.II – Estimate E[Z] with 95% Confidence Interval
*GitHub issue: #21*

In [ ]:
n = 10_000
rng_q5 = np.random.default_rng(0)
scores = die_experiment(n, rng_q5)
mean_score = np.mean(scores)
std_dev = np.std(scores, ddof=1)          # sample std
margin = 1.96 * std_dev / np.sqrt(n)      # 95% CI half-width
print(f"Estimated E[Z]          : {mean_score:.4f}")
print(f"95% CI                  : [{mean_score - margin:.4f}, {mean_score + margin:.4f}]")
print(f"Analytical E[Z] = 49/4  : {49/4:.4f}")

`die_experiment(10_000, rng_q5)` generates 10,000 scores to estimate $E[Z]$ and its uncertainty.

* `np.mean(scores)` — the point estimate $\bar{Z}$.
* `np.std(scores, ddof=1)` — the **sample** standard deviation with Bessel's correction (dividing by $N-1$ rather than $N$). The `ddof=1` matters here because we are estimating the *population* standard deviation $\sigma$ from a sample. Without it (`ddof=0`), the estimate is biased downward; the bias is negligible at $N=10{,}000$ but using `ddof=1` is correct practice regardless of sample size.
* `1.96 * std_dev / np.sqrt(n)` — the 95% CI half-width. This is the SEM scaled by the normal quantile: by the Central Limit Theorem, $\bar{Z}$ is approximately $\mathcal{N}(\mu, \sigma^2/N)$ for large $N$, so $\bar{Z} \pm 1.96\hat{\sigma}/\sqrt{N}$ covers the true mean with 95% probability. The factor 1.96 is the same used for the CV error bars in Section 1.
* The analytical value $49/4 = 12.25$ (derived in Q5.III) is printed for direct comparison. The expected gap between the simulation and analytical value is approximately $\pm 0.02$ at this sample size ($1.96 \sigma / \sqrt{10{,}000}$), which is consistent with the output.

### Q5.III – Analytical Derivation of E[Z]
*GitHub issue: #22*

**Step 1: Conditional expectation $E[Z \mid X = x]$**

The expected value of a single fair six-sided die roll:

$$E[Y] = \sum_{k=1}^{6} k \cdot \frac{1}{6} = \frac{1 + 2 + 3 + 4 + 5 + 6}{6}
= \frac{21}{6} = \frac{7}{2}$$

Given $X = x$, exactly $x$ dice are rolled, each with expected value $\frac{7}{2}$, each with $E[Y(i)] = \frac{7}{2}$, and $Y(i) = 0$ for $i > x$:

$$E[Z \mid X = x] = \sum_{i=1}^{6} E[Y(i) \mid X = x] = \sum_{i=1}^{x} \frac{7}{2} + \sum_{i=x+1}^{6} 0 = \frac{7x}{2}$$

**Step 2: Marginal expectation via total expectation**

$$E[Z] = \sum_{x=1}^{6} E[Z \mid X = x] \cdot P(X = x) = \sum_{x=1}^{6} \frac{7x}{2} \cdot \frac{1}{6} = \frac{7}{12} \sum_{x=1}^{6} x$$

$$= \frac{7}{12} \cdot (1 + 2 + 3 + 4 + 5 + 6) = \frac{7}{12} \cdot 21 =
\frac{147}{12} = \frac{49}{4} = 12.25$$

So the expected player score is $E[Z] = \dfrac{49}{4} = 12.25$.

---

---
#### Plain-Language Explanation of the Q5.III Derivation

---

**What are we trying to find?**

We want to know the long-run average score $Z$ a player achieves. This is the *expected value* $E[Z]$ — if you played the game thousands of times and averaged the scores, what number would you converge to?

---

**Why is the expected value of one die roll $\frac{7}{2} = 3.5$?**

A fair die lands on $\{1, 2, 3, 4, 5, 6\}$ with equal probability $\frac{1}{6}$ each. The expected value is the probability-weighted average of all outcomes:

$$E[Y] = \frac{1}{6}(1 + 2 + 3 + 4 + 5 + 6) = \frac{21}{6} = \frac{7}{2} = 3.5$$

Intuitively, 3.5 is exactly the midpoint of 1 and 6, which makes sense for a symmetric distribution.

---

**Step 1: Conditional expectation — fixing $X$ first**

The game has two stages, which makes it hard to compute $E[Z]$ directly. The standard technique is to first ask: *if I already knew the outcome of the first roll, what would my expected score be?*

This is called a **conditional expectation**, written $E[Z \mid X = x]$.

Given $X = x$ (the first roll came up $x$), the player rolls exactly $x$ more dice. Each of those dice independently has expected value $\frac{7}{2}$. Because **expected values add** (linearity of expectation — this holds even for random variables that depend on each other), the expected total of $x$ dice is simply $x$ times the expected value of one die:

$$E[Z \mid X = x] = \underbrace{\frac{7}{2} + \frac{7}{2} + \cdots + \frac{7}{2}}_{x \text{ terms}} = \frac{7x}{2}$$

Concretely:

| First roll $x$ | Subsequent dice rolled | $E[Z \mid X = x]$ |
|:-:|:-:|:-:|
| 1 | 1 die | $3.5$ |
| 2 | 2 dice | $7.0$ |
| 3 | 3 dice | $10.5$ |
| 4 | 4 dice | $14.0$ |
| 5 | 5 dice | $17.5$ |
| 6 | 6 dice | $21.0$ |

---

**Step 2: Removing the condition — the Law of Total Expectation**

We do not get to choose $X$; it is random. The **Law of Total Expectation** says we can find the overall $E[Z]$ by averaging the conditional expectations over all possible values of $X$, each weighted by how likely that value of $X$ is:

$$E[Z] = \sum_{x=1}^{6} E[Z \mid X = x] \cdot P(X = x)$$

Since the die is fair, every outcome has probability $P(X = x) = \frac{1}{6}$. Substituting $E[Z \mid X = x] = \frac{7x}{2}$:

$$E[Z] = \sum_{x=1}^{6} \frac{7x}{2} \cdot \frac{1}{6} = \frac{7}{12} \sum_{x=1}^{6} x = \frac{7}{12} \cdot 21 = \frac{49}{4} = 12.25$$

The sum $1 + 2 + 3 + 4 + 5 + 6 = 21$ is the same sum that gave us $E[Y] = \frac{7}{2}$ — it reappears here because $X$ also follows a uniform distribution over $\{1, \ldots, 6\}$.

---

**Sanity check against intuition**

The average first roll is $E[X] = 3.5$. On average, the player therefore rolls $3.5$ subsequent dice, each worth $3.5$ on average. So we would intuitively expect:

$$E[Z] \approx E[X] \cdot \frac{7}{2} = 3.5 \times 3.5 = 12.25$$

The formal derivation confirms this exactly. The simulation in Q5.II gives $\approx 12.23$, consistent with the analytical value of $12.25$ (the small difference is sampling noise).

---
## Tests
### `fruit_experiment` and `die_experiment` Test Suite

In [ ]:
print("=== fruit_experiment checks ===")

rng_t = np.random.default_rng(0)

boxes_t, fruits_t = fruit_experiment(1000, rng_t)

assert len(boxes_t)  == 1000, f"Expected 1000 box samples, got {len(boxes_t)}"
assert len(fruits_t) == 1000, f"Expected 1000 fruit samples, got {len(fruits_t)}"
print("  PASS  output arrays have length n")

assert set(boxes_t)  <= {'red', 'blue', 'yellow'}, f"Unexpected box values: {set(boxes_t)}"
assert set(fruits_t) <= {'apple', 'orange'},        f"Unexpected fruit values: {set(fruits_t)}"
print("  PASS  all box and fruit values are from the valid sets")

# Marginal box probability should be ~1/3 each (uniform box selection)
for box in ['red', 'blue', 'yellow']:
    p = np.mean(boxes_t == box)
    assert 0.28 < p < 0.38, f"P({box}) = {p:.3f}, expected ~0.333"
print("  PASS  box selection is approximately uniform (each ~1/3)")

# Reproducibility: same rng seed → same output
b1, f1 = fruit_experiment(50, np.random.default_rng(99))
b2, f2 = fruit_experiment(50, np.random.default_rng(99))
assert np.array_equal(b1, b2) and np.array_equal(f1, f2), \
    "Same rng seed should produce identical output"
print("  PASS  same rng seed produces reproducible output")

# Empirical P(yellow | apple) should be close to analytical 4/11
n_large = 200_000
b_large, f_large = fruit_experiment(n_large, np.random.default_rng(0))
apple_idx = f_large == 'apple'
p_yellow_given_apple = np.mean(b_large[apple_idx] == 'yellow')
assert abs(p_yellow_given_apple - 4/11) < 0.01, \
    f"P(yellow|apple) = {p_yellow_given_apple:.4f}, expected ~{4/11:.4f}"
print(f"  PASS  P(yellow|apple) ≈ {p_yellow_given_apple:.4f}  (analytical: {4/11:.4f})")

print("\n=== die_experiment checks ===")

scores_t = die_experiment(1000, np.random.default_rng(0))

assert scores_t.shape == (1000,), f"Expected shape (1000,), got {scores_t.shape}"
print("  PASS  output shape is (n_repetitions,)")

assert scores_t.min() >= 1,  f"Minimum score {scores_t.min()} should be >= 1"
assert scores_t.max() <= 36, f"Maximum score {scores_t.max()} should be <= 36"
print(f"  PASS  all scores are in the valid range [1, 36]  (observed: [{scores_t.min()}, {scores_t.max()}])")

# Reproducibility
s1 = die_experiment(100, np.random.default_rng(7))
s2 = die_experiment(100, np.random.default_rng(7))
assert np.array_equal(s1, s2), "Same seed should produce identical scores"
print("  PASS  same seed produces reproducible scores")

# E[Z] should converge to 49/4 = 12.25 for large n
n_large = 100_000
mean_z = np.mean(die_experiment(n_large, np.random.default_rng(0)))
assert abs(mean_z - 12.25) < 0.1, f"E[Z] = {mean_z:.4f}, expected ~12.25"
print(f"  PASS  empirical E[Z] ≈ {mean_z:.4f}  (analytical: 12.25)")

print("\nAll tests passed.")

### `Box` Test Suite

In [ ]:
print("=== Box checks ===")

# Use a fresh box so global state (red/blue/yellow) is untouched
b = Box("test")
assert b.total_fruits == 0, "New box should have 0 fruits"
print("  PASS  new box initialises with 0 total fruits")

b.add_fruit("apple", 3)
b.add_fruit("orange", 5)
assert b.get_fruit_count("apple")  == 3, "Expected 3 apples"
assert b.get_fruit_count("orange") == 5, "Expected 5 oranges"
assert b.total_fruits == 8,              "Expected total_fruits = 8"
print("  PASS  add_fruit updates counts and total_fruits correctly")

assert np.isclose(b.fruit_weights.sum(), 1.0), "Weights should sum to 1"
apple_idx = b.fruit_types.index("apple")
assert np.isclose(b.fruit_weights[apple_idx], 3/8), \
    f"Apple weight should be 3/8, got {b.fruit_weights[apple_idx]}"
print("  PASS  fruit_weights sum to 1 and reflect correct proportions")

# Adding to an existing fruit type accumulates
b.add_fruit("apple", 1)
assert b.get_fruit_count("apple") == 4, "Expected 4 apples after second add"
assert b.total_fruits == 9
print("  PASS  repeated add_fruit accumulates correctly")

# remove_fruit decrements count
b.remove_fruit("apple", 2)
assert b.get_fruit_count("apple") == 2, "Expected 2 apples after removal"
assert b.total_fruits == 7
print("  PASS  remove_fruit decrements count and total_fruits correctly")

# remove_fruit raises ValueError when count is insufficient
try:
    b.remove_fruit("apple", 99)
    assert False, "Should have raised ValueError"
except ValueError:
    pass
print("  PASS  remove_fruit raises ValueError when count is insufficient")

# get_fruit_count returns 0 for a fruit that was never added
assert b.get_fruit_count("banana") == 0
print("  PASS  get_fruit_count returns 0 for absent fruit")

# sample_fruit returns only valid fruit types
rng_b = np.random.default_rng(0)
samples = [b.sample_fruit(rng_b) for _ in range(200)]
assert set(samples) <= set(b.fruit_types), \
    f"Unexpected fruit type in samples: {set(samples) - set(b.fruit_types)}"
print("  PASS  sample_fruit returns only valid fruit types")

# Proportions: box has 2 apples and 5 oranges → apple rate ≈ 2/7 ≈ 0.286
apple_rate = samples.count("apple") / len(samples)
assert 0.15 < apple_rate < 0.45, \
    f"Apple sample rate {apple_rate:.3f} is outside plausible range for true rate {2/7:.3f}"
print(f"  PASS  sample_fruit respects proportions (apple rate {apple_rate:.3f} ≈ {2/7:.3f})")

print("\nAll tests passed.")